In [ ]:
# !pip uninstall openai
# !pip install -U openai

     ------------------------------------- 644.8/644.8 kB 10.1 MB/s eta 0:00:00
  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached jiter-0.9.0-cp39-cp39-win_amd64.whl (208 kB)
  Attempting uninstall: openai
    Found existing installation: openai 0.27.7
    Uninstalling openai-0.27.7:
      Successfully uninstalled openai-0.27.7



[notice] A new release of pip available: 22.2.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import openai
import os 

def create_prompt(product):
    client = openai.OpenAI(api_key="",)

    response = client.responses.create(
        model="gpt-4o",
        input=f"""
    Objective: Generate a list of search queries to prompt into tavily, each seach query seperated with commar, to retrieve web-scraped data for the {product}:

    Product Design Reviews (user/analyst feedback on product design)

    Product Improvement Suggestions (how to enhance the product)

    Functional Design Specifications (technical & usability specs)

    Competitor Information (similar products, market positioning)

    Constraints:

    Queries should be optimized for search engines/web scraping.

    Include both broad and specific keyword variations.

    Prioritize authoritative sources (e.g., industry blogs, whitepapers, forums, competitor websites).

    Do not put in website url

    Put everything in a single sentence, no breaklines
        

        """
    )

    return response.output_text.split(',')



In [31]:

from dotenv import load_dotenv
import requests
import pandas as pd

load_dotenv()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

def webscrape(product):
 
    url = "https://api.tavily.com/search"

    payload = {
        "query": product ,
        "topic": "general",
        "search_depth": "advanced",
        "chunks_per_source": 3,
        "max_results": 100,
        "time_range": None,
        "days": 3,
        "include_answer": True,
        "include_raw_content": False,
        "include_images": False,
        "include_image_descriptions": False,
        # "include_domains": ["https://www.reddit.com/", "https://www.pcgamer.com/","https://www.ign.com/news","https://www.reddit.com/r/GamingChairReviews","https://www.techgearlab.com","https://www.tomshardware.com","https://chairdeskexpert.com","https://www.buzzfeed.com" ],
        "include_domains": [],
        "exclude_domains": ["https://www.youtube.com/"]
    }
    headers = {
        "Authorization": "Bearer " + TAVILY_API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.request("POST", url, json=payload, headers=headers)

    response = response.json()
    # print(response)
    answer = response["answer"]
    # print(answer)
    results_dict = {}
    for result in response["results"]:
        results_dict[result["title"]] = {"url":result["url"], 
                                         "content":result["content"]}
    return answer, results_dict






# product = "Secretlab Titan Evo Lite gaming chair review"
# product = "gaming chair positive product "
# product = "Scrape online reviews for top gaming chairs. Extract common complaints on comfort, durability, and ergonomics. Focus on armrests, lumbar support, and materials."

# product ="Compare gaming chairs vs. ergonomic office chairs  on adjustability, breathability, and posture support. Highlight gaps in gaming chair designs."
# product ="Find technical specs for gaming chair materials (examples: PU leather, memory foam density, frame alloys). Include stress-test results or warranty claims."
# product ="Scrape user complaints about gaming chair assembly (tools, instructions, part alignment)"
product ="gaming chair design issues"

webscrape_result = webscrape(product)
print(webscrape_result)

('Gaming chair design often focuses on ergonomics to prevent long-term health issues. Lumbar support and adjustable features are key. Some chairs lack proper breathability and comfort.', {'Higher end chairs - for gaming / working from home - TechPowerUp': {'url': 'https://www.techpowerup.com/forums/threads/higher-end-chairs-for-gaming-working-from-home.312013/', 'content': 'Most gaming chairs are poorly designed because they are modeled after automobile seats in sports cars which have a completely different'}, 'Gaming Chair Weaknesses Unveiled: A Guide to Informed Buying ...': {'url': 'https://sihoooffice.com/blogs/guide/gaming-chair-weaknesses-unveiled-a-guide-to-informed-buying-decisions', 'content': 'Portability Challenges:\n  Gaming chairs are often designed to be sturdy and robust, but this can translate to limited portability. Unlike traditional office chairs with wheels, some gaming chairs can be heavy and cumbersome to move around. This lack of portability may be inconvenient f

In [26]:
webscrape_result

('Gaming chair design often focuses on ergonomics to prevent long-term health issues. Lumbar support and adjustable features are key. Some chairs lack proper breathability and comfort.',
 {'Higher end chairs - for gaming / working from home - TechPowerUp': {'url': 'https://www.techpowerup.com/forums/threads/higher-end-chairs-for-gaming-working-from-home.312013/',
   'content': 'Most gaming chairs are poorly designed because they are modeled after automobile seats in sports cars which have a completely different'},
  'Gaming Chair Weaknesses Unveiled: A Guide to Informed Buying ...': {'url': 'https://sihoooffice.com/blogs/guide/gaming-chair-weaknesses-unveiled-a-guide-to-informed-buying-decisions',
   'content': 'Portability Challenges:\n  Gaming chairs are often designed to be sturdy and robust, but this can translate to limited portability. Unlike traditional office chairs with wheels, some gaming chairs can be heavy and cumbersome to move around. This lack of portability may be incon

In [40]:
mehmeh1 = {}
for x in create_prompt("Secretlab titan evo lite"):

    _, results = webscrape(x)
    mehmeh1.update(results)


    

In [41]:
mehmeh1

{'Secretlab Titan EVO Lite Review - Dutchiee.TV': {'url': 'https://www.dutchiee.tv/news/secretlab-titan-evo-lite-review/',
  'content': 'The lumbar support system ensures that the spine is properly positioned in a manner that will not lead to tiredness or pains especially when one'},
 'Secretlab TITAN Evo Lite Gaming Chair Review - Game Rant': {'url': 'https://gamerant.com/secretlab-titan-evo-lite-chair-review/',
  'content': "Right out of the package and after a quick and painless building process, the TITAN Evo Lite looks almost exactly like the TITAN Evo, so much so that it even comes in some of the OG Evo’s most popular colors. As Secretlab explains, the chair features 95% of the TITAN Evo’s features, only at 20% less cost to gamers. And in our testing, the TITAN Evo Lite doesn’t feel like a downgrade at all. It is still an exceptional gaming chair that offers the comfort and customization that Secretlab is known [...] Secretlab TITAN Evo Lite Gaming Chair\n\nDesigned for exception

In [42]:
mehmeh1.values()

dict_values([{'url': 'https://www.dutchiee.tv/news/secretlab-titan-evo-lite-review/', 'content': 'The lumbar support system ensures that the spine is properly positioned in a manner that will not lead to tiredness or pains especially when one'}, {'url': 'https://gamerant.com/secretlab-titan-evo-lite-chair-review/', 'content': "Right out of the package and after a quick and painless building process, the TITAN Evo Lite looks almost exactly like the TITAN Evo, so much so that it even comes in some of the OG Evo’s most popular colors. As Secretlab explains, the chair features 95% of the TITAN Evo’s features, only at 20% less cost to gamers. And in our testing, the TITAN Evo Lite doesn’t feel like a downgrade at all. It is still an exceptional gaming chair that offers the comfort and customization that Secretlab is known [...] Secretlab TITAN Evo Lite Gaming Chair\n\nDesigned for exceptional comfort over long hours of sitting, the Secretlab TITAN Evo Lite combines the\nmost essential featu

In [ ]:
webscrape_result

In [21]:
for x in webscrape_result[1].keys():
    print(webscrape_result[1][x])

{'url': 'https://www.techpowerup.com/forums/threads/higher-end-chairs-for-gaming-working-from-home.312013/', 'content': 'Most gaming chairs are poorly designed because they are modeled after automobile seats in sports cars which have a completely different'}
{'url': 'https://sihoooffice.com/blogs/guide/gaming-chair-weaknesses-unveiled-a-guide-to-informed-buying-decisions', 'content': 'Portability Challenges:\n  Gaming chairs are often designed to be sturdy and robust, but this can translate to limited portability. Unlike traditional office chairs with wheels, some gaming chairs can be heavy and cumbersome to move around. This lack of portability may be inconvenient for users who need to shift their setup or store the chair when not in use. Considering the space and mobility requirements is crucial when choosing a gaming chair. [...] Durability Concerns:\n  Despite their often premium price tags, not all gaming chairs are created equal in terms of durability. Some lower-end models may b

In [47]:
# webscrape_result[1]["Secretlab Titan Evo Lite review: a more affordable gaming chair"]

In [43]:
mehmehult = pd.DataFrame(data=mehmeh1.values())

In [34]:
webscrape_result[1].values()

dict_values([{'url': 'https://www.t3.com/reviews/secretlab-titan-evo-lite-review', 'content': 'Why you can trust T3\n\n\n\n\nOur expert reviewers spend hours testing and comparing products and services so you can choose the best for you. Find out more about how we test.\n\nThe Secretlab Titan Evo Lite is a brand-new chair from one of the biggest names in gaming chairs. It is designed to offer “95% of the premium experience” of the Secretlab Titan Evo for “20% cheaper”. It’s a simple concept, but does it work? [...] The Secretlab Titan Evo Lite was officially released on 24 January 2024. The chair is available in five versions, including two leatherette and three SoftWeave colours and is priced from £389 / $449 directly from the Secretlab website. The chair also comes in an XL version for £459 /$499. [...] As with previous Secretlab chairs, the packaging for the Titan Evo Lite is beautifully done. Everything comes nicely packaged and there are both clear instructions and even a video gu

In [44]:
mehmehult.to_csv("gamingchairsss.csv",index=False)